# SQL-on-FHIR in the browser (JupyterLite / Pyodide) &mdash; #650

This notebook runs **entirely in your browser** (Python via Pyodide, no server-side
interpreter). It calls the HFS **`$sql-run`** endpoint and analyzes the result with
pandas + matplotlib.

For the fetch to reach `$sql-run`, this JupyterLite site must be served from the
**same origin as HFS** (e.g. mounted under a `/ui/notebook/` route). Then the
relative URL `/$sql-run` hits the running server and the browser sends the UI's
same-origin cookies — "data, not credentials".


In [ ]:
# pandas + matplotlib ship with Pyodide; import triggers a one-time load.
import json
import pandas as pd
import pyodide.http

VIEW = {
    "resourceType": "ViewDefinition",
    "status": "active",
    "resource": "Patient",
    "select": [
        {"column": [
            {"name": "id", "path": "id"},
            {"name": "gender", "path": "gender"},
            {"name": "birth_date", "path": "birthDate"},
            {"name": "active", "path": "active"},
        ]}
    ],
}
VIEW


In [ ]:
# Top-level await works in JupyterLite. POST the ViewDefinition to $sql-run.
df = None
try:
    resp = await pyodide.http.pyfetch(
        "/$sql-run?_format=json",
        method="POST",
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        body=json.dumps(VIEW),
    )
    if resp.status >= 400:
        print("HTTP", resp.status, await resp.string())
    else:
        rows = await resp.json()
        df = pd.DataFrame(rows)
        print(f"{len(df)} rows from $sql-run")
        display(df.head())
except Exception as e:
    print("Fetch failed. Serve this site under the HFS origin and seed Patients.\n ", e)


In [ ]:
import matplotlib.pyplot as plt

if df is not None and len(df):
    counts = df["gender"].value_counts()
    ax = counts.plot.bar(color="#2a78d6", figsize=(5, 3))
    ax.set_title("Patients by gender (in-browser, via $sql-run)")
    ax.set_ylabel("count")
    plt.tight_layout(); plt.show()
    print("Total patients:", len(df))
else:
    print("(no data — see the fetch cell above)")


## Notes

- **No server-side Python.** Everything above executed in the browser tab.
- **Offline / no-CDN.** The default `build.sh` output pulls Pyodide from
  `cdn.jsdelivr.net` at runtime, which the HFS no-CDN guard forbids. The vendored
  build (`--pyodide <tarball>`, ~463 MB) makes zero off-origin requests and passes
  — see the README.
- **Persistence.** JupyterLite stores notebooks in the browser's IndexedDB, which
  is per-browser-profile and shared across tenants on a shared machine — scope or
  disable it before using this near PHI.
